# MGTS Object ImageReviewer — обучение в Google Colab

Среда выполнения → Сменить среду → **GPU T4**. Затем запустите ячейки по порядку.

In [ ]:
!git clone -q -b ml-pipeline https://github.com/EfimovAN18/MGTS_Object_ImageReviewer.git
%cd /content/MGTS_Object_ImageReviewer
!pip install -q ultralytics roboflow
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow Private API Key: ')

In [ ]:
!python scripts/01_download_datasets.py
!python scripts/03_train_detector.py --epochs 25 --batch 32
!python scripts/04_train_classifier.py --epochs 10
!ls -la models

In [ ]:
import glob, random, cv2
from IPython.display import Image, display
from src.inspector import Inspector
insp = Inspector()
for p in random.sample(glob.glob('data/det/images/val/*manhole*.jpg'), 4):
    r, a = insp.analyze_image(cv2.imread(p))
    d = r.to_dict()
    print(d['status'], [(o['object_name'], o['status'], o['damage_prob']) for o in d['objects']])
    cv2.imwrite('/tmp/t.jpg', a)
    display(Image('/tmp/t.jpg', width=400))

In [ ]:
!zip -j -q mgts_models.zip models/detector.pt models/classifier.pt && ls -la mgts_models.zip
from google.colab import files
files.download('mgts_models.zip')